In [ ]:
import math
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from scipy.interpolate import Rbf
from scipy.interpolate import griddata

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

Using device: cpu


In [ ]:
class VolatilitySurfaceCNN(nn.Module):
    def __init__(self, param_bounds):
        super().__init__()
        num_parameters = len(param_bounds)

        self.register_buffer('param_mins', torch.tensor([b[0] for b in param_bounds], dtype=torch.float32))
        self.register_buffer('param_maxs', torch.tensor([b[1] for b in param_bounds], dtype=torch.float32))
        self.register_buffer('param_ranges', self.param_maxs - self.param_mins)

        self.conv_block = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ELU(),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ELU()
        )

        # CHANGE HERE: Added + 2 for Sentiment and Articles
        self.flattened_size = (32 * 8 * 11) + 2

        self.dense_block = nn.Sequential(
            nn.Linear(self.flattened_size, 256),
            nn.BatchNorm1d(256),
            nn.ELU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ELU(),
            nn.Linear(128, num_parameters),
            nn.Sigmoid()
        )

    # CHANGE HERE: Added 'context_data' as an input
    def forward(self, iv_surface, context_data):
        features = self.conv_block(iv_surface)
        features_flat = features.view(features.size(0), -1)

        # CHANGE HERE: Glue the 2 news numbers to the CNN features
        combined = torch.cat((features_flat, context_data), dim=1)

        out_0_to_1 = self.dense_block(combined)
        real_world_params = (out_0_to_1 * self.param_ranges) + self.param_mins

        return out_0_to_1, real_world_params



In [ ]:
def interpolate_surface_linear_nearest(day_data, value_col, LM_mesh, T_mesh):
    points = np.column_stack([
        day_data['LogMoneyness'].values,
        day_data['Tau'].values
    ])
    values = day_data[value_col].values

    xi = np.column_stack([LM_mesh.ravel(), T_mesh.ravel()])

    # linear interpolation first
    interp_linear = griddata(points, values, xi, method='linear')

    # fill any holes with nearest neighbor
    interp_nearest = griddata(points, values, xi, method='nearest')

    interp = np.where(np.isnan(interp_linear), interp_nearest, interp_linear)
    return interp.reshape(LM_mesh.shape)

In [ ]:
def GenerateGaussLaguerre(n):
    from math import factorial

    def nchoosek(n_, r_):
        return factorial(n_) / (factorial(r_) * factorial(n_ - r_))

    x, _ = np.polynomial.laguerre.laggauss(n)
    w = np.zeros(n)
    dL = np.zeros((n, len(x)))

    for j in range(len(x)):
        for k in range(n):
            dL[k, j] = (-1) ** (k + 1) / factorial(k) * nchoosek(n, k + 1) * x[j] ** k
        w[j] = 1 / x[j] / sum(dL[:, j]) ** 2
        w[j] = w[j] * np.exp(x[j])

    return (
        torch.tensor(x, dtype=torch.complex128, device=device).unsqueeze(0),
        torch.tensor(w, dtype=torch.complex128, device=device).unsqueeze(0)
    )


def JumpCF_PT(phi, lambdJ, muJ, sigmaJ, T):
    i = 1j
    term1 = -lambdJ * muJ * i * phi * T
    term2 = lambdJ * T * (
        (1 + muJ) ** (i * phi) * torch.exp(0.5 * sigmaJ**2 * i * phi * (i * phi - 1)) - 1
    )
    return torch.exp(term1 + term2)


def HestonCF_PT(phi, kappa, theta, sigma, v0, rho, T, S, r, q):
    i = 1j
    x = torch.log(S)
    a = kappa * theta
    u = -0.5
    b = kappa

    d = torch.sqrt((rho * sigma * i * phi - b) ** 2 - sigma**2 * (2 * u * i * phi - phi**2))
    c = (b - rho * sigma * i * phi - d) / (b - rho * sigma * i * phi + d)
    exp_dT = torch.exp(-d * T)

    D = ((b - rho * sigma * i * phi - d) / sigma**2) * ((1 - exp_dT) / (1 - c * exp_dT))
    G = (1 - c * exp_dT) / (1 - c)
    C = (r - q) * i * phi * T + a / sigma**2 * ((b - rho * sigma * i * phi - d) * T - 2 * torch.log(G))

    return torch.exp(C + D * v0 + i * phi * x)


def BatesCF_PT(phi, kappa, theta, sigma, v0, rho, lambdJ, muJ, sigmaJ, T, S, r, q):
    return HestonCF_PT(phi, kappa, theta, sigma, v0, rho, T, S, r, q) * JumpCF_PT(phi, lambdJ, muJ, sigmaJ, T)

In [ ]:
def SVprice_Batched(kappa, theta, sigma, v0, rho, lambdJ, muJ, sigmaJ, T, K, S, r, q, x_nodes, w_nodes):
    i = 1j

    kappa = kappa.unsqueeze(1)
    theta = theta.unsqueeze(1)
    sigma = sigma.unsqueeze(1)
    v0 = v0.unsqueeze(1)
    rho = rho.unsqueeze(1)
    lambdJ = lambdJ.unsqueeze(1)
    muJ = muJ.unsqueeze(1)
    sigmaJ = sigmaJ.unsqueeze(1)

    if not torch.is_tensor(T):
        T = torch.tensor(T, dtype=torch.float32, device=device)
    if not torch.is_tensor(K):
        K = torch.tensor(K, dtype=torch.float32, device=device)
    if not torch.is_tensor(S):
        S = torch.tensor(S, dtype=torch.float32, device=device)
    if not torch.is_tensor(r):
        r = torch.tensor(r, dtype=torch.float32, device=device)
    if not torch.is_tensor(q):
        q = torch.tensor(q, dtype=torch.float32, device=device)

    T = T.reshape(1, 1)
    K = K.reshape(1, 1)
    S = S.reshape(1, 1)
    r = r.reshape(1, 1)
    q = q.reshape(1, 1)

    phi = x_nodes

    f2 = BatesCF_PT(phi, kappa, theta, sigma, v0, rho, lambdJ, muJ, sigmaJ, T, S, r, q)
    int2 = (torch.exp(-i * phi * torch.log(K)) * f2 / (i * phi)).real

    phi_minus_i = phi - i
    f1_num = BatesCF_PT(phi_minus_i, kappa, theta, sigma, v0, rho, lambdJ, muJ, sigmaJ, T, S, r, q)
    f1_den = BatesCF_PT(
        torch.tensor([[-1j]], dtype=torch.complex128, device=device),
        kappa, theta, sigma, v0, rho, lambdJ, muJ, sigmaJ, T, S, r, q
    )
    int1 = (torch.exp(-i * phi * torch.log(K)) * f1_num / (i * phi * f1_den)).real

    P1 = 0.5 + (1 / math.pi) * torch.sum(w_nodes * int1, dim=1)
    P2 = 0.5 + (1 / math.pi) * torch.sum(w_nodes * int2, dim=1)

    price = (
        S.squeeze(1) * torch.exp(-q.squeeze(1) * T.squeeze(1)) * P1
        - K.squeeze(1) * torch.exp(-r.squeeze(1) * T.squeeze(1)) * P2
    )

    price = price.real.to(torch.float32)
    return torch.clamp(price, min=1e-7)

In [ ]:
def create_iv_and_price_grids_from_raw(source_csv, max_days=None):
    df = pd.read_csv(source_csv, low_memory=False)

    required = ['Trade Date', 'LogMoneyness', 'Tau', 'Implied Volatility', 'S', 'daily_sentiment', 'articles_per_day']
    for col in required:
        if col not in df.columns:
            raise ValueError(f"Missing column {col} in {source_csv}")

    if 'Mid' in df.columns:
        price_col = 'Mid'
    elif 'OptionPrice' in df.columns:
        price_col = 'OptionPrice'
    else:
        raise ValueError("Raw file needs either 'Mid' or 'OptionPrice'.")

    # NEW: Scale Article counts to 0-1 range so they match Sentiment scale
    art_min = df['articles_per_day'].min()
    art_max = df['articles_per_day'].max()
    df['articles_scaled'] = (df['articles_per_day'] - art_min) / (art_max - art_min)

    df['Trade Date'] = pd.to_datetime(df['Trade Date'])

    tau_grid = [0.1, 0.3, 0.6, 0.9, 1.2, 1.5, 1.8, 2.0]
    lm_grid = [-0.5, -0.4, -0.3, -0.2, -0.1, 0.0, 0.1, 0.2, 0.3, 0.4, 0.5]
    LM_mesh, T_mesh = np.meshgrid(lm_grid, tau_grid)

    iv_surfaces = []
    price_surfaces = []
    context_data = [] # NEW: To store [Sentiment, Articles]

    unique_dates = sorted(df['Trade Date'].unique())
    print(f"Found {len(unique_dates)} days of raw market data...")

    used_days = 0
    skipped_days = 0

    for date in unique_dates:
        if max_days is not None and used_days >= max_days:
            break

        day_data = df[df['Trade Date'] == date].copy()
        day_data = day_data.dropna(subset=['LogMoneyness', 'Tau', 'Implied Volatility', price_col, 'S'])

        if len(day_data) < 15:
            skipped_days += 1
            continue

        day_data['NormalizedPrice'] = day_data[price_col] / day_data['S']
        day_data = day_data[day_data['NormalizedPrice'] >= 0.0]

        if len(day_data) < 15:
            skipped_days += 1
            continue

        low_q = day_data['NormalizedPrice'].quantile(0.01)
        high_q = day_data['NormalizedPrice'].quantile(0.99)
        day_data['NormalizedPrice'] = day_data['NormalizedPrice'].clip(lower=low_q, upper=high_q)

        try:
            iv_interp = interpolate_surface_linear_nearest(day_data, 'Implied Volatility', LM_mesh, T_mesh)
            price_interp = interpolate_surface_linear_nearest(day_data, 'NormalizedPrice', LM_mesh, T_mesh)

            if np.isnan(iv_interp).any() or np.isinf(iv_interp).any():
                skipped_days += 1
                continue
            if np.isnan(price_interp).any() or np.isinf(price_interp).any():
                skipped_days += 1
                continue

            # NEW: Grab Sentiment and Scaled Articles for this day
            daily_sent = day_data['daily_sentiment'].iloc[0]
            daily_arts = day_data['articles_scaled'].iloc[0]

            iv_surfaces.append(iv_interp)
            price_surfaces.append(price_interp)
            context_data.append([daily_sent, daily_arts]) # Store the pair

            used_days += 1

        except Exception:
            skipped_days += 1
            continue

    print(f"Used days: {used_days}")
    print(f"Skipped days: {skipped_days}")

    iv_tensor = torch.tensor(np.array(iv_surfaces), dtype=torch.float32).unsqueeze(1)
    price_tensor = torch.tensor(np.array(price_surfaces), dtype=torch.float32).unsqueeze(1)
    context_tensor = torch.tensor(np.array(context_data), dtype=torch.float32) # NEW

    return iv_tensor, price_tensor, context_tensor

In [ ]:
class RealWorldFineTuner:
    def __init__(self, model, x_mean, x_std, device):
        self.model = model
        self.device = device
        self.x_mean = x_mean.to(device)
        self.x_std = x_std.to(device)

        self.maturities = torch.tensor(
            [0.1, 0.3, 0.6, 0.9, 1.2, 1.5, 1.8, 2.0],
            dtype=torch.float32,
            device=device
        )
        self.log_moneyness = torch.tensor(
            [-0.5, -0.4, -0.3, -0.2, -0.1, 0.0, 0.1, 0.2, 0.3, 0.4, 0.5],
            dtype=torch.float32,
            device=device
        )

        # normalized pricing scale
        self.S = torch.tensor(1.0, dtype=torch.float32, device=device)
        self.r = torch.tensor(0.045, dtype=torch.float32, device=device)
        self.q = torch.tensor(0.011, dtype=torch.float32, device=device)

        self.x_nodes, self.w_nodes = GenerateGaussLaguerre(32)

    def calculate_model_price_surface(self, params):
        model_prices = []

        for T in self.maturities:
            row_prices = []
            F = self.S * torch.exp((self.r - self.q) * T)

            for lm in self.log_moneyness:
                K = F * torch.exp(-lm)

                prices = SVprice_Batched(
                    params[:, 0], params[:, 1], params[:, 2], params[:, 3],
                    params[:, 4], params[:, 5], params[:, 6], params[:, 7],
                    T, K, self.S, self.r, self.q, self.x_nodes, self.w_nodes
                )

                row_prices.append(prices.unsqueeze(1))

            row_prices = torch.cat(row_prices, dim=1)
            model_prices.append(row_prices.unsqueeze(1))

        surface = torch.cat(model_prices, dim=1).unsqueeze(1)
        return surface.to(torch.float32)

    def fine_tune(self, real_data_loader, epochs=50, lr=5e-6):
        self.model.train()
        optimizer = torch.optim.Adam(self.model.parameters(), lr=lr)
        criterion = nn.HuberLoss(delta=0.1)

        loss_history = []
        printed_debug = False
        best_loss = float('inf')

        print(f"Starting Fine-Tuning with Sentiment/Articles on {len(real_data_loader.dataset)} market days...")

        for epoch in range(epochs):
            total_loss = 0.0

            # This loop must be indented inside the 'for epoch' loop
            for market_iv_surface, market_price_surface, context_data in real_data_loader:
                market_iv_surface = market_iv_surface.to(self.device, dtype=torch.float32)
                market_price_surface = market_price_surface.to(self.device, dtype=torch.float32)
                context_data = context_data.to(self.device, dtype=torch.float32)

                scaled_input = ((market_iv_surface - self.x_mean) / self.x_std).to(torch.float32)

                optimizer.zero_grad()

                # Pass both the IV surface and the News Context to the model
                _, predicted_params = self.model(scaled_input, context_data)

                model_generated_price_surface = self.calculate_model_price_surface(predicted_params).to(torch.float32)

                if not printed_debug:
                    print("market min/max:", market_price_surface.min().item(), market_price_surface.max().item())
                    print("model  min/max:", model_generated_price_surface.min().item(), model_generated_price_surface.max().item())
                    printed_debug = True

                loss = criterion(model_generated_price_surface, market_price_surface)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                optimizer.step()

                total_loss += loss.item()

            # These lines must be aligned with the START of the 'for market_iv' loop
            avg_loss = total_loss / len(real_data_loader)
            loss_history.append(avg_loss)

            if avg_loss < best_loss:
                best_loss = avg_loss
                torch.save(self.model.state_dict(), "/content/Bates_Weights_Sentiment_Best.pth")

            print(f"Epoch {epoch + 1} | Huber Price Loss: {avg_loss:.6f}")

        return loss_history

In [ ]:
import pandas as pd
import torch
from torch.utils.data import TensorDataset, random_split

def recover_mean_std_from_bates_csv(csv_path, grid_shape=(8, 11)):
    df = pd.read_csv(csv_path)

    iv_cols = [col for col in df.columns if col.startswith('IV_T')]
    X_flat = df[iv_cols].values
    X_surfaces = X_flat.reshape(-1, 1, grid_shape[0], grid_shape[1])

    X_tensor = torch.tensor(X_surfaces, dtype=torch.float32)
    dummy_y = torch.zeros(len(X_tensor), 1)
    dataset = TensorDataset(X_tensor, dummy_y)

    total_size = len(dataset)
    train_size = int(0.7 * total_size)
    val_size = int(0.2 * total_size)
    test_size = total_size - train_size - val_size

    train_dataset, _, _ = random_split(
        dataset,
        [train_size, val_size, test_size],
        generator=torch.Generator().manual_seed(42)
    )

    X_train = train_dataset.dataset.tensors[0][train_dataset.indices]
    X_train_mean = X_train.mean().item()
    X_train_std = X_train.std().item()

    return X_train_mean, X_train_std

csv_path = "/content/Bates_IV_Surface_Data.csv"   # change path if needed
x_mean, x_std = recover_mean_std_from_bates_csv(csv_path)

print("Recovered X_MEAN_VAL =", x_mean)
print("Recovered X_STD_VAL  =", x_std)

Recovered X_MEAN_VAL = 0.32219305634498596
Recovered X_STD_VAL  = 0.09585768729448318


In [ ]:
MODEL_WEIGHTS_PATH = '/content/BatesNNWeights.pth'
RAW_FILE_PATH = '/content/cleanerSPXData_2010_merged.csv'

X_MEAN_VAL = x_mean
X_STD_VAL = x_std

bates_bounds = [
    (0.1, 3.0),    # kappa
    (0.01, 0.15),  # theta
    (0.1, 0.8),    # sigma
    (0.01, 0.15),  # v0
    (-0.9, -0.1),  # rho
    (0.0, 2.0),    # lambdaJ
    (-0.2, 0.05),  # muJ
    (0.01, 0.20)   # sigmaJ
]


model = VolatilitySurfaceCNN(param_bounds=bates_bounds).to(device)

try:
    model.load_state_dict(torch.load(MODEL_WEIGHTS_PATH, map_location=device))
    print("Pre-trained weights loaded successfully.")
except Exception as e:
    print(f"Error loading weights: {e}")

X_mean = torch.tensor(X_MEAN_VAL, dtype=torch.float32).to(device)
X_std = torch.tensor(X_STD_VAL, dtype=torch.float32).to(device)

real_iv_tensor, real_price_tensor, real_context_tensor = create_iv_and_price_grids_from_raw(RAW_FILE_PATH, max_days=None)

# And update the loader right below it:
real_loader = DataLoader(
    TensorDataset(real_iv_tensor, real_price_tensor, real_context_tensor),
    batch_size=16,
    shuffle=True
)


print(f"Final Count: {len(real_iv_tensor)} surfaces ready for medium fine-tuning test.")

model_before = copy.deepcopy(model)

tuner = RealWorldFineTuner(model, X_mean, X_std, device)
loss_history = tuner.fine_tune(real_loader, epochs=10, lr=5e-6)

save_path = "/content/Bates_Weights_PriceSpace_MediumTest.pth"
torch.save(model.state_dict(), save_path)
print(f"Success! Adapted weights saved to: {save_path}")

Error loading weights: Error(s) in loading state_dict for VolatilitySurfaceCNN:
	size mismatch for dense_block.0.weight: copying a param with shape torch.Size([256, 2816]) from checkpoint, the shape in current model is torch.Size([256, 2818]).
Found 253 days of raw market data...
Used days: 250
Skipped days: 3
Final Count: 250 surfaces ready for medium fine-tuning test.
Starting Fine-Tuning with Sentiment/Articles on 250 market days...
market min/max: 0.000847919553052634 4.732529163360596
model  min/max: 1.0000000116860974e-07 0.4254358410835266
Epoch 1 | Huber Price Loss: 0.036344
Epoch 2 | Huber Price Loss: 0.036109
Epoch 3 | Huber Price Loss: 0.036097
Epoch 4 | Huber Price Loss: 0.036196
Epoch 5 | Huber Price Loss: 0.036152
Epoch 6 | Huber Price Loss: 0.036037
Epoch 7 | Huber Price Loss: 0.036005
Epoch 8 | Huber Price Loss: 0.036119
Epoch 9 | Huber Price Loss: 0.035989
Epoch 10 | Huber Price Loss: 0.035853
Success! Adapted weights saved to: /content/Bates_Weights_PriceSpace_MediumT

In [ ]:
real_iv_tensor, real_price_tensor, real_context_tensor = create_iv_and_price_grids_from_raw(RAW_FILE_PATH)

real_loader = DataLoader(
    TensorDataset(real_iv_tensor, real_price_tensor, real_context_tensor), # Added tensor here
    batch_size=16,
    shuffle=True
)

print(f"Final Count: {len(real_iv_tensor)} surfaces ready for longer fine-tuning test.")

# --- FINAL VISUALIZATION AND SANITY CHECK ---

# 1. Grab a sample day from our data (index 0)
sample_iv = real_iv_tensor[0:1].to(device)
sample_price = real_price_tensor[0:1].to(device)
sample_context = real_context_tensor[0:1].to(device) # NEW: Get news data for this sample

# 2. Set models to evaluation mode
tuner_before = RealWorldFineTuner(model_before, X_mean, X_std, device)
tuner_before.model.eval()

tuner_after = RealWorldFineTuner(model, X_mean, X_std, device)
tuner_after.model.eval()

# 3. Generate Predictions
with torch.no_grad():
    # Scale the IV surface just like we do in training
    scaled_input = ((sample_iv - X_mean) / X_std).to(torch.float32)

    # Pass BOTH the surface and the context data to the model
    _, params_before = tuner_before.model(scaled_input, sample_context)
    pred_price_before = tuner_before.calculate_model_price_surface(params_before)

    _, params_after = tuner_after.model(scaled_input, sample_context)
    pred_price_after = tuner_after.calculate_model_price_surface(params_after)

# 4. Prepare data for plotting (move back to CPU/Numpy)
market_surface = sample_price.squeeze().cpu().numpy()
before_surface = pred_price_before.squeeze().cpu().numpy()
after_surface = pred_price_after.squeeze().cpu().numpy()

# 5. Calculate Error and Improvement
before_error = np.abs(before_surface - market_surface)
after_error = np.abs(after_surface - market_surface)
improvement = before_error - after_error

# 6. Create the 6-panel Plot
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

im0 = axes[0, 0].imshow(market_surface, aspect='auto')
axes[0, 0].set_title("Market Price Surface")
plt.colorbar(im0, ax=axes[0, 0])

im1 = axes[0, 1].imshow(before_surface, aspect='auto')
axes[0, 1].set_title("Model Before Fine-Tuning")
plt.colorbar(im1, ax=axes[0, 1])

im2 = axes[0, 2].imshow(after_surface, aspect='auto')
axes[0, 2].set_title("Model After Fine-Tuning")
plt.colorbar(im2, ax=axes[0, 2])

im3 = axes[1, 0].imshow(before_error, aspect='auto')
axes[1, 0].set_title("Absolute Error Before")
plt.colorbar(im3, ax=axes[1, 0])

im4 = axes[1, 1].imshow(after_error, aspect='auto')
axes[1, 1].set_title("Absolute Error After")
plt.colorbar(im4, ax=axes[1, 1])

im5 = axes[1, 2].imshow(improvement, aspect='auto')
axes[1, 2].set_title("Error Improvement")
plt.colorbar(im5, ax=axes[1, 2])

plt.suptitle(f"Sentiment: {sample_context[0,0]:.2f} | Articles: {sample_context[0,1]:.2f}")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(loss_history) + 1), loss_history, marker='o')
plt.xlabel("Epoch")
plt.ylabel("Price Fit Loss")
plt.title("Fine-Tuning Loss by Epoch")
plt.grid(True)
plt.show()

In [ ]:
sample_iv = real_iv_tensor[0:1].to(device)
sample_price = real_price_tensor[0:1].to(device)

tuner_before = RealWorldFineTuner(model_before, X_mean, X_std, device)
tuner_before.model.eval()

tuner_after = RealWorldFineTuner(model, X_mean, X_std, device)
tuner_after.model.eval()

with torch.no_grad():
    scaled_input = ((sample_iv - X_mean) / X_std).to(torch.float32)

    _, params_before = tuner_before.model(scaled_input)
    pred_price_before = tuner_before.calculate_model_price_surface(params_before)

    _, params_after = tuner_after.model(scaled_input)
    pred_price_after = tuner_after.calculate_model_price_surface(params_after)

market_surface = sample_price.squeeze().cpu().numpy()
before_surface = pred_price_before.squeeze().cpu().numpy()
after_surface = pred_price_after.squeeze().cpu().numpy()

before_error = np.abs(before_surface - market_surface)
after_error = np.abs(after_surface - market_surface)
improvement = before_error - after_error

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

im0 = axes[0, 0].imshow(market_surface, aspect='auto')
axes[0, 0].set_title("Market Price Surface")
plt.colorbar(im0, ax=axes[0, 0])

im1 = axes[0, 1].imshow(before_surface, aspect='auto')
axes[0, 1].set_title("Model Before Fine-Tuning")
plt.colorbar(im1, ax=axes[0, 1])

im2 = axes[0, 2].imshow(after_surface, aspect='auto')
axes[0, 2].set_title("Model After Fine-Tuning")
plt.colorbar(im2, ax=axes[0, 2])

im3 = axes[1, 0].imshow(before_error, aspect='auto')
axes[1, 0].set_title("Absolute Error Before")
plt.colorbar(im3, ax=axes[1, 0])

im4 = axes[1, 1].imshow(after_error, aspect='auto')
axes[1, 1].set_title("Absolute Error After")
plt.colorbar(im4, ax=axes[1, 1])

im5 = axes[1, 2].imshow(improvement, aspect='auto')
axes[1, 2].set_title("Error Improvement")
plt.colorbar(im5, ax=axes[1, 2])

plt.tight_layout()
plt.show()